In [2]:
# %pip install pandas openpyxl

import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd


# RMO (Rijksmuseum van Oudheden)

In [10]:

BASE_URL = "http://api.rmo.nl:17521/"
IMAGE_BASE = "https://www.rmo.nl/imageproxy/"
QUERIES = {
    "nehalennia": None,
    "colijnsplaat": ("Vindplaats", "colijnsplaat"),
}
PAGE_SIZE = 100
SEP = " | "

OUT_DIR = Path("out")
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Fetch
The API takes its parameters as a path (`action=get&command=search&...`), not a query string. Raw XML pages are cached to `out/raw/` so reruns don't hit the server.

In [11]:
def fetch_page(query, start, end, use_cache=True):
    cache = RAW_DIR / f"{query}_{start}-{end}.xml"
    if use_cache and cache.exists():
        return ET.fromstring(cache.read_bytes())
    params = f"action=get&command=search&query={urllib.parse.quote(query)}&range={start}-{end}&fields=*"
    with urllib.request.urlopen(BASE_URL + params, timeout=60) as resp:
        body = resp.read()
    cache.write_bytes(body)
    time.sleep(0.5)  # be polite
    return ET.fromstring(body)


def harvest(query):
    first = fetch_page(query, 1, PAGE_SIZE)
    total = int(first.findtext("request/count"))
    print(f"{total} hits for {query!r}")
    pages = [first]
    for start in range(PAGE_SIZE + 1, total + 1, PAGE_SIZE):
        pages.append(fetch_page(query, start, min(start + PAGE_SIZE - 1, total)))
    records = [rec for page in pages for rec in page.iter("record")]
    print(f"harvested {len(records)} records")
    return records


def matches(rec, field_filter):
    if field_filter is None:
        return True
    field, value = field_filter
    return any(value.lower() in (el.text or "").lower() for el in rec.findall(f"data/RMO_XML/{field}"))


records = {}  # recordnumber -> (record, [queries it matched])
for query, field_filter in QUERIES.items():
    kept = [rec for rec in harvest(query) if matches(rec, field_filter)]
    if field_filter:
        print(f"kept {len(kept)} with {field_filter[0]} containing {field_filter[1]!r}")
    for rec in kept:
        key = rec.findtext("data/RMO_XML/recordnumber")
        records.setdefault(key, (rec, []))[1].append(query)
print(f"{len(records)} unique records")

276 hits for 'nehalennia'
harvested 276 records
386 hits for 'colijnsplaat'
harvested 386 records
kept 386 with Vindplaats containing 'colijnsplaat'
411 unique records


## Flatten

In [12]:
def image_url(filename):
    stem, _, ext = filename.rpartition(".")
    return f"{IMAGE_BASE}{ext.lower()}/{stem}"


def flatten(rec):
    data = rec.find("data/RMO_XML")
    row, images = {}, []
    for el in data:
        text = (el.text or "").strip()
        if el.tag == "image":
            images.append({"filename": text, "image_url": image_url(text), **el.attrib})
            continue
        row.setdefault(el.tag, []).append(text)
    row = {k: SEP.join(v) for k, v in row.items()}
    row["image_count"] = len(images)
    row["image_url"] = images[0]["image_url"] if images else None
    row["image_filenames"] = SEP.join(i["filename"] for i in images)
    row["image_copyright"] = SEP.join(dict.fromkeys(i.get("copyright", "") for i in images))
    for i in images:
        i["Inventarisnummer"] = row.get("Inventarisnummer")
        i["recordnumber"] = row.get("recordnumber")
    return row, images


rows, image_rows = [], []
for rec, queries in records.values():
    row, imgs = flatten(rec)
    row["matched_queries"] = SEP.join(queries)
    rows.append(row)
    image_rows.extend(imgs)

df = pd.DataFrame(rows)
# put the most useful columns first
front = ["Inventarisnummer", "Titel", "image_url", "matched_queries", "Objectnaam", "Materiaal", "Periode",
         "datebegin", "dateend", "Vindplaats", "Afmetingenlabel", "Inscripties", "Beschrijving"]
df = df[[c for c in front if c in df] + [c for c in df if c not in front]]

df_images = pd.DataFrame(image_rows)
df_images = df_images[["Inventarisnummer", "recordnumber"] + [c for c in df_images if c not in ("Inventarisnummer", "recordnumber")]]

print(df.shape, df_images.shape)
print(df["matched_queries"].value_counts())
df.head()

(411, 31) (1228, 6)
matched_queries
nehalennia | colijnsplaat    251
colijnsplaat                 135
nehalennia                    25
Name: count, dtype: int64


,Inventarisnummer,Titel,image_url,matched_queries,Objectnaam,Materiaal,Periode,datebegin,dateend,Vindplaats,...,europeana_provider,europeana_object,europeana_isShownBy,europeana_isShownAt,dc_identifier,dc_rights,imagetijdlijn,image_count,image_filenames,image_copyright
0,i 1970/12.7,Nehalennia-altaar,https://www.rmo.nl/imageproxy/tiff/019732,nehalennia | colijnsplaat,altaar,kalksteen,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",...,Rijksmuseum van Oudheden,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,https://hdl.handle.net/21.12126/137816,http://www.rmo.nl/collectie/zoeken?object=i 19...,Creative Commons By-licensie: bron: Rijksmuseu...,119271.jpg,10,019732.tiff | 019474.tiff | 019730.tiff | 0201...,Creative Commons By-licensie: bron: Rijksmuseu...
1,i 1970/12.9,Nehalennia-altaar,https://www.rmo.nl/imageproxy/tiff/019527,nehalennia | colijnsplaat,altaar,kalksteen,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",...,Rijksmuseum van Oudheden,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,https://hdl.handle.net/21.12126/137817,http://www.rmo.nl/collectie/zoeken?object=i 19...,Creative Commons By-licensie: bron: Rijksmuseu...,130512.jpg,16,019527.tiff | 019427.tiff | 019528.tiff | 0196...,Creative Commons By-licensie: bron: Rijksmuseu...
2,i 1970/12.24,Nehalennia-altaar,https://www.rmo.nl/imageproxy/tiff/019423,nehalennia | colijnsplaat,altaar,kalksteen,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",...,Rijksmuseum van Oudheden,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,https://hdl.handle.net/21.12126/137818,http://www.rmo.nl/collectie/zoeken?object=i 19...,Creative Commons By-licensie: bron: Rijksmuseu...,119424.jpg,10,019423.tiff | 019601.tiff | 019666.tiff | 0197...,Creative Commons By-licensie: bron: Rijksmuseu...
3,i 1970/12.16,Nehalennia-altaar,https://www.rmo.nl/imageproxy/tiff/015782,nehalennia | colijnsplaat,altaar,kalksteen,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",...,Rijksmuseum van Oudheden,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,https://hdl.handle.net/21.12126/137819,http://www.rmo.nl/collectie/zoeken?object=i 19...,Creative Commons By-licensie: bron: Rijksmuseu...,015781.tiff,4,015782.tiff | 015781.tiff | 019696.tiff | 0196...,Creative Commons By-licensie: bron: Rijksmuseu...
4,i 1970/12.1,Nehalennia-altaar,https://www.rmo.nl/imageproxy/tiff/019737,nehalennia | colijnsplaat,altaar,kalksteen,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",...,Rijksmuseum van Oudheden,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,http://img.rmo.nl/imageproxy/proxy.aspx?maxwid...,https://hdl.handle.net/21.12126/138005,http://www.rmo.nl/collectie/zoeken?object=i 19...,Creative Commons By-licensie: bron: Rijksmuseu...,015427.tiff,28,019737.tiff | 019743.tiff | 019476.tiff | 0194...,Creative Commons By-licensie: bron: Rijksmuseu...


## Export

In [ ]:
EXCEL_CELL_LIMIT = 32767
too_long = df.map(lambda v: isinstance(v, str) and len(v) > EXCEL_CELL_LIMIT)
if too_long.any().any():
    print("truncating over-long cells in:", list(df.columns[too_long.any()]))
    df = df.map(lambda v: v[:EXCEL_CELL_LIMIT] if isinstance(v, str) else v)

LINK_COLUMNS = ["image_url", "europeana_isShownAt"]


def make_links_clickable(ws):
    header = [c.value for c in ws[1]]
    for name in LINK_COLUMNS:
        if name not in header:
            continue
        col = header.index(name) + 1
        for (cell,) in ws.iter_rows(min_row=2, min_col=col, max_col=col):
            if cell.value:
                cell.hyperlink = cell.value
                cell.style = "Hyperlink"


out_file = OUT_DIR / f"rmo_{'_'.join(QUERIES)}.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as xw:
    df.to_excel(xw, sheet_name="records", index=False)
    df_images.to_excel(xw, sheet_name="images", index=False)
    for ws in xw.sheets.values():
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        make_links_clickable(ws)

wrote /Users/Minh.Tran/Documents/rijks/cit_hackalod/fetch/out/rmo_nehalennia_colijnsplaat.xlsx
